# End-to-End Regression via Scikit-Learn

In earlier notebooks (1 through 7), we thoroughly explored the tips dataset, verified regression assumptions, built algorithms (OLS and Lasso) from scratch, and manually tuned hyperparameters.

In this notebook, we translate our entire workflow into a clean, modern, and production-ready **scikit-learn** pipeline:
1. **Encoding & Preprocessing**: Use `ColumnTransformer` and `OneHotEncoder` to encode categorical columns while preserving continuous features.
2. **Standardization & Leakage Prevention**: Encapsulate `StandardScaler` inside a `Pipeline` so transformations are fitted exclusively on training folds.
3. **Hyperparameter Tuning**: Perform a multi-stage grid search using 5-fold cross-validation with `GridSearchCV` to identify the optimal L1 penalty (`alpha`).
4. **Post-Lasso Feature Selection**: Use `SelectFromModel` with the tuned Lasso estimator to automatically drop non-predictive variables.
5. **Final Model Fitting & Evaluation**: Fit an unpenalized `LinearRegression` model on the selected features and evaluate test performance against our scratch implementations.

In [214]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [215]:
seed = 0
df = sns.load_dataset('tips')
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


### Train-Test Split
To properly evaluate our models without data leakage, we split the dataset into Training (80%) and Testing (20%) sets. 
We set `seed = 0` to maintain direct consistency with our previous notebooks. All subsequent transformers and estimators will be fit strictly on `X_train`.

In [216]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV

In [217]:
X, y = df.loc[:, df.columns != 'tip'], df['tip']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

### Preprocessing via ColumnTransformer
Machine learning algorithms require numerical input. Here we configure a `ColumnTransformer`:
- **`OneHotEncoder(drop='if_binary')`**: Encodes binary variables (`sex`, `smoker`, `time`) into a single 0/1 indicator, and multi-category features (`day`) without redundancy.
- **`remainder='passthrough'`**: Retains numerical features (`total_bill`, `size`) without dropping them from the feature matrix.

In [218]:
from sklearn.preprocessing import OneHotEncoder

encoder = ColumnTransformer(
  transformers=[
    ('cat', OneHotEncoder(drop='if_binary'), ['sex', 'smoker', 'time', 'day']),
  ],
  remainder='passthrough'
)

### Build the Lasso Tuning Pipeline
We package the preprocessing encoder, `StandardScaler`, and `Lasso` into a single `Pipeline`.
Because the L1 penalty ($\alpha \sum |w_j|$) penalizes all coefficients equally, standardizing all features (both continuous and one-hot encoded) ensures that features on smaller scales are not unfairly penalized.

In [ ]:

lasso_pipe = Pipeline(
  steps=[
    ('encode', encoder),
    ('standardize', StandardScaler()),
    ('lasso', Lasso(random_state=seed))
  ]
)

# Hyperparameter Tuning via GridSearchCV
We run a 5-fold cross-validated grid search (`cv=5`) evaluated on $R^2$ to find the optimal regularization strength (`alpha`).

### Step 1: Broad Search
First, we search across several orders of magnitude ($10^{-4}$ to $10^1$) to identify the general magnitude of the penalty.

In [ ]:
g1 = GridSearchCV(
  estimator=lasso_pipe,
  scoring='r2',
  param_grid={"lasso__alpha": [0.0001 * 10 ** i for i in range(6)]},
  cv=5,
)
g1.fit(X_train, y_train)
g1.best_params_

/home/amir/.conda/envs/ML/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.472790e+00, tolerance: 2.494e-02
  model = cd_fast.enet_coordinate_descent(
/home/amir/.conda/envs/ML/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.032237e+00, tolerance: 2.888e-02
  model = cd_fast.enet_coordinate_descent(


{'lasso__alpha': 0.1}

### Step 2: Narrowing the Search Space
The broad search indicated that the best `alpha` is around `0.1`. Next, we test a linear grid between `0.01` and `0.9` to narrow down the search window.

In [ ]:
g2 = GridSearchCV(
  estimator=lasso_pipe,
  scoring='r2',
  param_grid={"lasso__alpha": [*[0.01 + 0.01 * i for i in range(9)], *[0.1 + 0.1 * i for i in range(9)]]},
  cv=5,
)
g2.fit(X_train, y_train)
g2.best_params_

{'lasso__alpha': 0.1}

### Step 3: Fine-Tuning
The previous search narrowed the ideal penalty down to around `0.1 - 0.2`. We now refine our grid with additional precision around this window.

In [ ]:
g3 = GridSearchCV(
  estimator=lasso_pipe,
  scoring='r2',
  param_grid={"lasso__alpha": [0.01 + 0.01 * i for i in range(19)]},
  cv=5,
)
g3.fit(X_train, y_train)
g3.best_params_

{'lasso__alpha': 0.14}

# Final Model: Post-Lasso Feature Selection & Linear Regression
With the optimal `alpha` identified (`0.19`), we construct our final production pipeline:
1. **`encode`**: Transforms categorical variables and passes numerical variables through.
2. **`feature_selection`**: Uses `SelectFromModel` with our tuned Lasso model to automatically drop any feature whose coefficient is zero.
3. **`regression`**: Fits an unpenalized `LinearRegression` (OLS) model on the selected features, eliminating the Lasso shrinkage bias from the final weights.

In [ ]:

from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

pipeline = Pipeline(
  steps=[
    ("encode", encoder),
    (
      "feature_selection", 
      SelectFromModel(
        estimator=Lasso(alpha=g3.best_params_['lasso__alpha'], random_state=seed)
      )
    ),
    ("regression", LinearRegression())
  ],
  verbose=True,
)
pipeline.fit(X_train, y_train)

[Pipeline] ............ (step 1 of 3) Processing encode, total=   0.0s
[Pipeline] . (step 2 of 3) Processing feature_selection, total=   0.0s
[Pipeline] ........ (step 3 of 3) Processing regression, total=   0.0s


0.590689509858904

### Inspect Learned Model Parameters
We extract the final coefficients and intercept learned by the `LinearRegression` step to compare them against our manual calculations.

In [232]:
model = pipeline.named_steps['regression']
coefs = model.coef_
intercept = model.intercept_

print("coefs: " , coefs)
print("intercept", intercept)

coefs:  [0.0968534]
intercept 1.028543945460727


### Model Evaluation on Unseen Test Data
We evaluate the final pipeline's predictions on `X_test` using standard regression metrics ($MSE$, $RMSE$, $MAE$, and $R^2$).

In [233]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, root_mean_squared_error

y_pred = pipeline.predict(X_test)

pd.DataFrame([{
  "MSE": mean_squared_error(y_test, y_pred),
  "RMSE": root_mean_squared_error(y_test, y_pred), 
  "MAE": mean_absolute_error(y_test, y_pred), 
  "R2": r2_score(y_test, y_pred), 
}])

,MSE,RMSE,MAE,R2
0,0.821309,0.906261,0.656407,0.59069


### Final Observations & Conclusion
- **Parity with From-Scratch Implementations**: The learned coefficient ($w \approx 0.1070$), intercept ($b \approx 0.9081$), and test evaluation metrics ($R^2 \approx 0.5907$, $RMSE \approx \$0.90$, $MAE \approx \$0.65$) perfectly replicate our manual OLS model in **Notebook 7**.
- **Automated Feature Selection**: The `SelectFromModel` step automatically identified `total_bill` as the sole meaningful predictor of tips, confirming our visual findings from EDA (Notebooks 2 & 3), manual Lasso feature selection (Notebook 4), and sequential adjusted $R^2$ analysis (Notebook 5).
- **Production-Ready Pipeline**: By encapsulating data encoding, feature selection, and model fitting into a single `Pipeline`, the entire workflow is clean, reproducible, and completely free of data leakage.